## **Machine Learning For Drug Design and Discovery**
## `Orgranized By:` Panacea Research Center
## `Topic:` Fundamentals of Python Part: (II)
## `Author:` Pritom Kundu || MD Ahad Ali
## Researcher and Trainer || Panacea Research Center, Rajshahi, Bangladeesh
## `What is CHEMBL Data Base`
The ChEMBL database is one of the most important public resources in drug discovery, medicinal chemistry, and bioinformatics. Below is a clear, consistent, and detail-focused explanation covering what it is, how it works, and why scientists use it.ChEMBL is a large, open scientific database that stores information about bioactive molecules — chemical compounds that interact with biological targets such as proteins, enzymes, or receptors.

## `In This Note Book`
We will explore a hands on go through and deep dive into the chembl data base and learn what data are there in the data base. How to select
- Target Protein
- Find Chembl ID
- Criteria of Selection
- Data Preprocessing
- Data Curation
- Data Clealing

## `We will learn it by doing it:`
In This project we will work with the Target protein `CDK2` Cyclin Dependent Kinase (CDK) is a protein on the surface of cells that helps them grow and divide. When the gene coding for this protein mutates or becomes overactive, it acts like a stuck "on" switch, leading to uncontrolled cell proliferation—the hallmark of cancer. we will aquire the data explore it and do preprocessing and EDA

### Step 1: `Installation`
To fetch data from CHEMBL web server we need to install some dependencies which is `chembl_webresource_client` so we will install it

In [ ]:
# Installation of chembl_webresource_client
!pip install chembl_webresource_client

### Step 2 `Importing The Needed Libraries`
In this project we will use libraries Pandas and Numpy

In [ ]:
import numpy as np
import pandas as pd
from chembl_webresource_client.new_client import new_client

### Step 3 `The Target Protein Search (Data Curation)`
In this step of the project we will search are target Protein we need.
### `Important Criteria to keep in mind`
- We must need a single protein for ML Project
- The assay type should be Binding Assay **B**
- We need `IC50` Data Which means `standard_type` = IC50`  

In [ ]:
# Target Query: EGFR
target = new_client.target  # Gets the target value from new_client object and saves it into the target variable.
target_query = target.search("CDK2") # Searches for the target protein EGFR

In [ ]:
# Exploring How many Data and Targets and Store Them in a Data Frame
targets = pd.DataFrame.from_dict(target_query)
targets





,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,CDK2/CDK4,14.0,False,CHEMBL3038517,"[{'accession': 'P11802', 'component_descriptio...",PROTEIN FAMILY,9606
1,[],Mus musculus,CDK2/CDK4,14.0,False,CHEMBL4106185,"[{'accession': 'P30285', 'component_descriptio...",PROTEIN FAMILY,10090
2,[],Mus musculus,CDK2/CDK9,14.0,False,CHEMBL4106186,"[{'accession': 'Q99J95', 'component_descriptio...",PROTEIN FAMILY,10090
3,"[{'xref_id': 'CPX-2006', 'xref_name': 'Cyclin ...",Homo sapiens,CDK2/Cyclin A2,12.0,False,CHEMBL3038469,"[{'accession': 'P24941', 'component_descriptio...",PROTEIN COMPLEX,9606
4,"[{'xref_id': 'CPX-2005', 'xref_name': 'Cyclin ...",Homo sapiens,CDK2/Cyclin A1,12.0,False,CHEMBL3038470,"[{'accession': 'P24941', 'component_descriptio...",PROTEIN COMPLEX,9606
5,[],Homo sapiens,CDK2/Cyclin O,12.0,False,CHEMBL4106152,"[{'accession': 'P24941', 'component_descriptio...",PROTEIN COMPLEX,9606
6,"[{'xref_id': 'CPX-2016', 'xref_name': 'Cyclin ...",Homo sapiens,CDK2/Cyclin-E2,12.0,False,CHEMBL4523633,"[{'accession': 'P24941', 'component_descriptio...",PROTEIN COMPLEX,9606
7,[],Homo sapiens,CDK2/Cyclin-Y,12.0,False,CHEMBL4523634,"[{'accession': 'P24941', 'component_descriptio...",PROTEIN COMPLEX,9606
8,[],Homo sapiens,CDK2/Cyclin C,12.0,False,CHEMBL4888454,"[{'accession': 'P24941', 'component_descriptio...",PROTEIN COMPLEX,9606
9,[],Homo sapiens,Cyclin-dependent kinase 2,11.0,False,CHEMBL301,"[{'accession': 'P24941', 'component_descriptio...",SINGLE PROTEIN,9606


In [ ]:
# To Show How Many
print(targets.shape)

(30, 9)


### Step 3.1 `Selecting The appropiate Tartet from The target_query`
Here we will take the target according to out need which corresponds to the human EGFR, IC50, Assay:B, Single Protein

In [ ]:
selected_target = targets.target_chembl_id[9]
selected_target

'CHEMBL301'

### Step3.2 `Extracting Bioactivity Data Only`
**Now we will keep only the Bioactivity Data which is basically standard_type = IC50 and Discard the rest and make a pandas Data Frame**

In [ ]:
activity = new_client.activity
bioactivity = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")


In [ ]:
# Creating The Data Frame
df_1 = pd.DataFrame.from_dict(bioactivity)
df_1.head(2)

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,105376,[],CHEMBL660784,Inhibitory concentration against cyclin-depend...,B,None,None,BAO_0000190,...,Homo sapiens,Cyclin-dependent kinase 2,9606,None,None,IC50,uM,UO_0000065,None,36.0
1,None,None,106512,[],CHEMBL661128,Inhibitory concentration against cyclin-depend...,B,None,None,BAO_0000190,...,Homo sapiens,Cyclin-dependent kinase 2,9606,None,None,IC50,nM,UO_0000065,None,2.0


###Step 3.3 `Duplicate and Missing value Removal`
**This Step will Keep only the Unique Values and remove the duplicates or missing value if column (standard_value) is empty for a spacific Compound**


In [ ]:
df_2 = df_1.standard_type.unique()
df_3 = df_1[df_1['standard_value'].notna()]
df_4 = df_1[df_1['standard_relation'] == '='] # Corrected: assuming you want to filter for '='
print(df_4.shape)

(2779, 46)


### Step 3.4 `Saving The Clean File in CSV`

In [ ]:
df_clean = df_4.to_csv("01 Clean_Bioactivity_Data_CDK2.csv")

### Step 4 `Data Pre Process`
Here we will pre process our curated data keep the needed column as of our need and make it rady for Explanatory Data Analysis (EDA)

### Step 4.1 `Leveling The Compound`
**In this step we will level the compound as active, inactive or intermidiate we are considering**
- standard_value >= 10000 as inactive
- standard_value < = 100 as active
- standard_value outside this two criteria as intermidiate



In [ ]:
# Creating an empty list
bioactivity_class = []

# Iterating each molecule with for loop
for i in df_4.standard_value:
  if float(i) >= 10000:
    bioactivity_class.append("inactive")
  elif float(i) <= 1000:
    bioactivity_class.append("active")
  else:
    bioactivity_class.append("intermediate")

In [ ]:
# Iterate the molecule_chembl_id to a list
molecule_id = []
for i in df_4.molecule_chembl_id:
  molecule_id.append(i)

In [ ]:
# Iterate standard_value to a list
standard_value = []
for i in df_4.standard_value:
  standard_value.append(i)

In [ ]:
# Iterate canonical_smiles to a list
canonical_smiles = []
for i in df_4.canonical_smiles:
  canonical_smiles.append(i)

In [ ]:
# Combine the 4 lists into a dataframe
data_tuples = list(zip(molecule_id, canonical_smiles, bioactivity_class, standard_value))
df_5 = pd.DataFrame( data_tuples,  columns=['molecule_chembl_id', 'canonical_smiles', 'bioactivity_class',  'standard_value'])
print("Preprocessed Data Here")
df_5

Preprocessed Data Here


,molecule_chembl_id,canonical_smiles,bioactivity_class,standard_value
0,CHEMBL101052,COc1ccc(Nc2ccnc(Nc3ccc(OCC(O)CN(C)C)cc3)n2)cc1,inactive,36000.0
1,CHEMBL260416,O=C(Nc1n[nH]c2nc(-c3ccc(O)c(Br)c3)ccc12)C1CC1,active,2.0
2,CHEMBL84944,CC(C)C(CO)Nc1nc(NCc2cccc(O)c2)c2ncn(C(C)C)c2n1,active,30.0
3,CHEMBL419720,CN(C)CC(O)COc1ccc(Nc2ncc(Br)c(N(CCCC(F)(F)F)c3...,active,200.0
4,CHEMBL260929,O=C(Nc1n[nH]c2cc(-c3ccc[nH]3)ccc12)C1CC1,active,497.0
...,...,...,...,...
2774,CHEMBL4208172,CCN1CCN(Cc2ccc(Nc3ncc(F)c(-c4cc(F)c5nc6n(c5c4)...,active,51.1
2775,CHEMBL6064669,CCN1CCC(c2ccc(Nc3ncc(F)c(-c4cc(F)c5nc6n(c5c4)C...,active,114.0
2776,CHEMBL6064669,CCN1CCC(c2ccc(Nc3ncc(F)c(-c4cc(F)c5nc6n(c5c4)C...,active,120.0
2777,CHEMBL5748748,CC1(C)CCc2nc3c(F)cc(-c4nc(Nc5ccc6c(n5)CCN(C(=O...,active,17.6


### Step 4.2 `Exporting The Explanatry CSV file`
We will save the data that is preprocessed in variable df_5

In [ ]:
df_5.to_csv("02 preprocessed_Bioactivity_Data_CDK2.csv")
